# KYC Thesis — Model 2: Face Embedding (Verification)

**What this notebook does, in order:**
1. Installs dependencies
2. Loads a pretrained face embedding model
3. Evaluates on LFW pairs (ROC + EER)
4. Runs a lightweight PTQ test (optional)
5. Exports to ONNX
6. Attempts TFLite export
7. Saves results to Google Drive

**Runtime:** ~1–2 hours on Colab T4 GPU

**Before running:** Runtime → Change runtime type → T4 GPU


## Cell 1 — Install Dependencies

In [ ]:
%%capture
!pip install numpy==1.26.4 pandas==2.2.2 scikit-learn==1.4.2 facenet-pytorch==2.5.3 timm==0.9.16 onnx==1.15.0 onnxruntime==1.17.0 onnx-tf tensorflow

import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## Cell 2 — Imports

In [ ]:
import os
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
from torchvision import transforms

from facenet_pytorch import InceptionResnetV1, fixed_image_standardization
from sklearn.metrics import roc_curve

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')


## Cell 3 — Mount Google Drive

All outputs (metrics, plots, exports) are saved here.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR     = Path('/content/drive/MyDrive/kyc_thesis')
EXPORT_DIR   = BASE_DIR / 'models' / 'exports'
RESULTS_DIR  = BASE_DIR / 'results' / 'face_embedding'

for d in [EXPORT_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Directory structure created:')
print(f'  Exports:  {EXPORT_DIR}')
print(f'  Results:  {RESULTS_DIR}')


## Cell 4 — Load Face Embedding Model

Using InceptionResnetV1 pretrained on VGGFace2 (widely used baseline).
If you have MobileFaceNet weights, swap the model here.

In [ ]:
model = InceptionResnetV1(pretrained='vggface2').eval().to(DEVICE)
print('Model loaded ✅')


## Cell 5 — Load LFW Pairs

We use sklearn's LFW pairs dataset for quick verification evaluation.

In [ ]:
from sklearn.datasets import fetch_lfw_pairs

lfw_pairs = fetch_lfw_pairs(subset='test', color=True, resize=1.0)
pairs = lfw_pairs.pairs  # shape: (N, 2, H, W, C)
labels = lfw_pairs.target  # 1 = same, 0 = different
print(f'LFW pairs: {pairs.shape[0]}')


## Cell 6 — Preprocess + Embed Pairs

We resize to 160×160 and standardize with Facenet's fixed standardization.

In [ ]:
preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
    transforms.Lambda(fixed_image_standardization),
])

def embed_batch(batch):
    batch = batch.to(DEVICE)
    with torch.no_grad():
        emb = model(batch)
    return F.normalize(emb, p=2, dim=1).cpu()

batch_size = 64
all_scores = []

for i in range(0, len(pairs), batch_size):
    batch_pairs = pairs[i:i+batch_size]
    imgs1 = torch.stack([preprocess(img) for img in batch_pairs[:, 0]])
    imgs2 = torch.stack([preprocess(img) for img in batch_pairs[:, 1]])

    emb1 = embed_batch(imgs1)
    emb2 = embed_batch(imgs2)

    scores = F.cosine_similarity(emb1, emb2).numpy()
    all_scores.extend(scores)

all_scores = np.array(all_scores)
print('Embeddings complete ✅')


## Cell 7 — ROC + EER (Verification Quality)

In [ ]:
fpr, tpr, thresholds = roc_curve(labels, all_scores)
fnr = 1 - tpr
eer_idx = np.nanargmin(np.absolute((fnr - fpr)))
eer = (fpr[eer_idx] + fnr[eer_idx]) / 2
eer_threshold = thresholds[eer_idx]

print(f'EER: {eer*100:.2f}%')
print(f'EER threshold: {eer_threshold:.4f}')

plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, label='ROC')
plt.plot([0,1], [0,1], '--', color='gray')
plt.scatter([fpr[eer_idx]], [tpr[eer_idx]], color='red', label=f'EER={eer*100:.2f}%')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.title('LFW ROC Curve')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig(RESULTS_DIR / 'lfw_roc.png', dpi=120, bbox_inches='tight')
plt.show()

metrics = {
    'eer': float(eer),
    'eer_threshold': float(eer_threshold),
    'pairs': int(len(labels))
}
with open(RESULTS_DIR / 'lfw_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved ✅')


## Cell 8 — INT8 PTQ (Optional)

Note: embedding models can be sensitive to quantisation.

In [ ]:
from torch.quantization import quantize_dynamic

model_int8 = quantize_dynamic(model.cpu(), {torch.nn.Linear}, dtype=torch.qint8)
model_int8.eval()
print('INT8 PTQ model ready (Linear layers only)')


## Cell 9 — Export to ONNX

In [ ]:
import onnx
import onnxruntime as ort

onnx_path = EXPORT_DIR / 'face_embedder.onnx'

dummy = torch.randn(1, 3, 160, 160)
torch.onnx.export(
    model.cpu(), dummy, str(onnx_path),
    opset_version=12,
    input_names=['input'],
    output_names=['embedding'],
    dynamic_axes={'input': {0: 'batch'}, 'embedding': {0: 'batch'}}
)

onnx_model = onnx.load(str(onnx_path))
onnx.checker.check_model(onnx_model)

sess = ort.InferenceSession(str(onnx_path))
out = sess.run(None, {'input': dummy.numpy()})
print(f'ONNX exported ✅  Output shape: {out[0].shape}')


## Cell 10 — Export to TFLite (Optional)

This may fail depending on conversion support; document results either way.

In [ ]:
import subprocess

def onnx_to_tflite(onnx_path, tflite_path):
    import tensorflow as tf
    saved_model_dir = str(tflite_path).replace('.tflite', '_saved_model')
    result = subprocess.run([
        'python', '-m', 'onnx_tf.main', 'convert',
        '-i', str(onnx_path),
        '-o', saved_model_dir
    ], capture_output=True, text=True)
    if result.returncode != 0:
        print(f'Conversion error: {result.stderr[:500]}')
        return None

    converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
    tflite_model = converter.convert()
    with open(tflite_path, 'wb') as f:
        f.write(tflite_model)
    print(f'TFLite exported ✅  {tflite_path.name}')
    return tflite_path

tflite_path = EXPORT_DIR / 'face_embedder.tflite'
onnx_to_tflite(onnx_path, tflite_path)


## Cell 11 — Summary

In [ ]:
summary = {
    'model': 'InceptionResnetV1 (VGGFace2)',
    'eer': metrics['eer'],
    'eer_threshold': metrics['eer_threshold'],
    'pairs': metrics['pairs'],
}
with open(RESULTS_DIR / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('Summary saved ✅')
